READ ANALYSIS XLSX

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os
%matplotlib inline
os.getcwd()
file = '../Scenario_Analysis.xlsx'
df = pd.read_excel(file, sheet_name='CO2Duals')
df = df.loc[:, ~df.columns.str.contains(r'\.\d+$')]
df = df.set_index(df.columns[0])
hours = df.index
start_date = pd.Timestamp("2019-01-01 00:00")
dt_index = start_date + pd.to_timedelta(hours-1, unit="h")
df.index = dt_index
df.index.name = "datetime"
df.index.freq = 'h'


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.collections import PolyCollection

scenarios = ["Baseline","NoStorageCO2","NoStorageALL","H2Sales","CO2Abundance"]

df_long = df[scenarios].melt(var_name="Scenario", value_name="CO2_price").dropna()

idle_eps = 1e-9
idle_hours = (df_long.assign(is_idle=lambda d: d["CO2_price"].abs() <= idle_eps)
                    .groupby("Scenario")["is_idle"].sum()
                    .reindex(scenarios).fillna(0).astype(int))

stats = (df_long.groupby("Scenario")["CO2_price"]
               .agg(med="median", mean="mean")
               .reindex(scenarios))

# ✅ Put typography into seaborn theme so it doesn't override you
sns.set_theme(
    style="white",
    rc={
        "font.family": "DejaVu Sans",
        "font.size": 11,
        "axes.titlesize": 12,
        "axes.labelsize": 11,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "axes.titleweight": "bold",   # match your reference look
        "text.usetex": False,         # keep mathtext consistent
    }
)

fig, ax = plt.subplots(figsize=(8, 4.8))

sns.violinplot(
    data=df_long, x="Scenario", y="CO2_price",
    order=scenarios,
    inner="box", cut=0,
    density_norm="width",
    bw_adjust=0.9,
    linewidth=0.9,
    color="#5A9EE2",
    ax=ax
)

for coll in ax.collections:
    if isinstance(coll, PolyCollection):
        coll.set_alpha(0.9)
        coll.set_edgecolor("#666666")
        coll.set_linewidth(0.9)

x_positions = np.arange(len(scenarios))
ax.scatter(
    x_positions, stats["mean"].values,
    marker="D", s=30,
    facecolor="white", edgecolor="black",
    linewidth=0.8, zorder=5
)

y_annot = 126
for i, scen in enumerate(scenarios):
    ax.text(
        i, y_annot,
        f"med={stats.loc[scen,'med']:.0f}\n"
        f"mean={stats.loc[scen,'mean']:.0f}\n"
        f"zero-price={idle_hours.loc[scen]} hrs",
        ha="center", va="top", fontsize=9
    )

# ax.set_title(r"CO$_2$ Shadow Price Distribution (8760 hours)")
ax.set_xlabel("")
ax.set_ylabel(r"CO$_2$ Shadow Price (EUR/t)")
ax.set_ylim(0, 130)
ax.grid(False)

for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(0.9)

plt.tight_layout()
plt.savefig("CO2_ShadowPrice_Violin.pdf", bbox_inches="tight", dpi=600)
plt.show()

AVERAGE CO2 DUAL VALUE

In [ ]:
import matplotlib.pyplot as plt

selected_scenarios = [
    "Baseline",
    "NoStorageCO2",
    "NoStorageALL",
    "H2Sales",
    "CO2Abundance",
]

scenario_means = df.mean()[selected_scenarios]

fig, ax = plt.subplots(figsize=(8,4))

scenario_means.plot(
    kind='bar',
    ax=ax,
    color="#5A9EE2",
    edgecolor='black',
    linewidth=0.3
)

ax.set_ylabel("Average CO₂ Dual (EUR/t)", fontsize=12)
# ax.set_title("Average CO₂ Dual Value per Scenario")

# Grid behind bars
ax.grid(axis='y', alpha=0.3, linestyle='-')
ax.set_axisbelow(True)

# X and Y ticks
ax.set_xticklabels(ax.get_xticklabels(), rotation=0, ha='center')

max_val = int(scenario_means.max())
tick_step = 10  # same spacing as before

ax.set_yticks(range(0, max_val + 2*tick_step, tick_step))
ax.set_ylim(0, 90)

for i, value in enumerate(scenario_means):
    ax.text(
        i,                         # x-position
        value + (value * 0.01),                # y-position slightly above bar
        f"{value:.0f}",           # formatted label
        ha='center', va='bottom', fontsize=9
    )

plt.tight_layout()
plt.savefig("DualCO2Average.pdf", format="pdf", bbox_inches="tight")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# -----------------------------
# Scenarios
# -----------------------------
scenarios = ["Baseline", "NoStorageCO2", "NoStorageALL", "H2Sales", "CO2Abundance"]
scenario_labels = ["Baseline", "NoStorageCO2", "NoStorageALL", "H2Sales", "CO2Abundance"]

# -----------------------------
# Data
# -----------------------------
data_all = [df[s].dropna().values for s in scenarios]
data_pos = [v[v > 0] for v in data_all]
zero_share = np.array([(v == 0).mean() * 100 for v in data_all])

means_pos = np.array([np.mean(v) if len(v) else np.nan for v in data_pos])
medians_pos = np.array([np.median(v) if len(v) else np.nan for v in data_pos])
zero_hours = np.array([(v == 0).sum() for v in data_all])

x = np.arange(1, len(scenarios) + 1)

# -----------------------------
# Figure layout
# -----------------------------
fig, (ax1, ax2) = plt.subplots(
    2, 1,
    figsize=(9.2, 6.4),
    sharex=True,
    gridspec_kw={"height_ratios": [4.2, 1.2]}
)

# -----------------------------
# Top panel: violin plot
# -----------------------------
vp = ax1.violinplot(
    data_pos,
    positions=x,
    widths=0.50,
    showmeans=False,
    showmedians=False,
    showextrema=False
)

# Violin styling with borders
for body in vp["bodies"]:
    body.set_facecolor("#5b9bd5")   # keep/change if you want
    body.set_edgecolor("#4d4d4d")
    body.set_linewidth(0.9)
    body.set_alpha(0.8)

# Mean markers
ax1.scatter(
    x,
    means_pos,
    marker="D",
    s=34,
    color="black",
    alpha=0.85,
    zorder=3,
    label="Mean"
)

# Mean labels
for xi, m in zip(x, means_pos):
    if np.isfinite(m):
        ax1.annotate(
            f"{m:.0f}",
            (xi, m),
            xytext=(0, 6),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=9
        )

ax1.set_ylabel("CO$_2$ Dual Value (EUR/t)", fontsize=12)
ax1.set_title("CO$_2$ Dual Distribution (positive hours) + zero-hour share", fontsize=13, pad=10)

ax1.grid(axis="y", linestyle="--", linewidth=0.6, alpha=0.45)
ax1.set_axisbelow(True)

# Keep borders like your reference figure
for spine in ax1.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(0.9)

ax1.legend(frameon=True, loc="upper right", fontsize=10)

# Y-limits
all_pos_concat = np.concatenate([v for v in data_pos if len(v) > 0])
ymin = max(0, np.floor(all_pos_concat.min() / 5) * 5 - 5)
ymax = np.ceil(all_pos_concat.max() / 5) * 5 + 5
ax1.set_ylim(ymin, ymax)

# -----------------------------
# Bottom panel: zero-share bars
# -----------------------------
bars = ax2.bar(
    x,
    zero_share,
    width=0.50,
    color="#1f77b4",
    edgecolor="#4d4d4d",
    linewidth=0.8,
    alpha=0.95
)

for xi, z in zip(x, zero_share):
    ax2.annotate(
        f"{z:.1f}%",
        (xi, z),
        xytext=(0, 3),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=9
    )

ax2.set_ylabel("Zero hours (%)", fontsize=12)
ax2.grid(axis="y", linestyle="--", linewidth=0.6, alpha=0.35)
ax2.set_axisbelow(True)

for spine in ax2.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(0.9)

ax2.set_xticks(x)
ax2.set_xticklabels(scenario_labels, fontsize=10)

ymax = max(15, zero_share.max() * 1.25)  # more headroom
ax2.set_ylim(0, ymax)

# -----------------------------
# Final formatting
# -----------------------------
fig.subplots_adjust(
    left=0.10,
    right=0.97,
    top=0.92,
    bottom=0.11,
    hspace=0.07
)

plt.savefig("DualCO2_Violin_PositivePlusZeroShare_clean.pdf", dpi=600, bbox_inches="tight")
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.collections import PolyCollection

scenarios = ["Baseline","NoStorageCO2","NoStorageALL","H2Sales","CO2Abundance"]

df_long = df[scenarios].melt(var_name="Scenario", value_name="CO2_price").dropna()

idle_eps = 1e-9
idle_hours = (df_long.assign(is_idle=lambda d: d["CO2_price"].abs() <= idle_eps)
                    .groupby("Scenario")["is_idle"].sum()
                    .reindex(scenarios).fillna(0).astype(int))

stats = (df_long.groupby("Scenario")["CO2_price"]
               .agg(med="median", mean="mean")
               .reindex(scenarios))

sns.set_theme(
    style="white",
    rc={
        "font.family": "DejaVu Sans",
        "font.size": 11,
        "axes.titlesize": 12,
        "axes.labelsize": 11,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "axes.titleweight": "bold",
        "text.usetex": False,
    }
)

# ✅ Wider figure
fig, ax = plt.subplots(figsize=(11.5, 4.8))

sns.violinplot(
    data=df_long, x="Scenario", y="CO2_price",
    order=scenarios,
    inner="box", cut=0,
    density_norm="width",
    bw_adjust=0.9,
    linewidth=0.9,
    color="#5A9EE2",
    ax=ax
)

for coll in ax.collections:
    if isinstance(coll, PolyCollection):
        coll.set_alpha(0.9)
        coll.set_edgecolor("#666666")
        coll.set_linewidth(0.9)

# ✅ Mean marker
x_positions = np.arange(len(scenarios))
ax.scatter(
    x_positions, stats["mean"].values,
    marker="D", s=30,
    facecolor="white", edgecolor="black",
    linewidth=0.8, zorder=6
)

# ✅ Raw values as a "side scatter" (to the right of each violin)
rng = np.random.default_rng(42)
side_offset = 0.45
x_jitter = 0.03

for i, scen in enumerate(scenarios):
    y = df_long.loc[df_long["Scenario"] == scen, "CO2_price"].to_numpy()
    x = (i + side_offset) + rng.uniform(-x_jitter, x_jitter, size=y.size)

    ax.scatter(
        x, y,
        s=6,
        alpha=0.3,
        color="dimgrey",
        linewidths=0,
        zorder=4,
    )

# ✅ Annotations (relative to top)
ax.set_ylim(-2, 130)
y_annot = ax.get_ylim()[1] - 3
for i, scen in enumerate(scenarios):
    ax.text(
        i, y_annot,
        f"med={stats.loc[scen,'med']:.0f}\n"
        f"mean={stats.loc[scen,'mean']:.0f}\n"
        f"Idle={idle_hours.loc[scen]} hrs",
        ha="center", va="top", fontsize=9
    )

ax.set_xlim(-0.5, len(scenarios)-0.5 + 0.05)

ax.set_title(r"CO$_2$ Shadow Price Distribution (8760 hours)")
ax.set_xlabel("")
ax.set_ylabel(r"CO$_2$ Shadow Price (EUR/t)")
ax.grid(False)

for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(0.9)

plt.tight_layout()
plt.savefig("CO2_ShadowPrice_Violin_withPoints.pdf", bbox_inches="tight", dpi=600)
plt.show()

PLOT HOURLY DUALS ALONG THEIR MOVING AVERAGE

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import NullLocator
import matplotlib.colors as mc
import colorsys

# helper function: darken a color nicely
def darken(color, amount=0.55):
    try:
        c = mc.cnames[color]
    except:
        c = color
    h, l, s = colorsys.rgb_to_hls(*mc.to_rgb(c))
    return colorsys.hls_to_rgb(h, max(0, min(1, l * amount)), s)


def plot_scenarios(df, scenarios, scenario_colors, start=None, end=None, ma_window=168):

    data = df[scenarios]
    if start or end:
        data = data.loc[start:end]

    fig, ax = plt.subplots(figsize=(14, 5))

    for scen in scenarios:
        base_color = scenario_colors[scen]
        ma_color = darken(base_color, 0.8)

        # Hourly line (faint)
        ax.plot(
            data.index,
            data[scen],
            color=base_color,
            alpha=0.25,
            label=scen
        )

        # Weekly moving average (bold)
        ma = data[scen].rolling(window=ma_window, min_periods=1).mean()
        ax.plot(
            data.index,
            ma,
            color=ma_color,
            label=f"{ma_window}h MA ({scen})"
        )

    ax.set_xlabel("")
    ax.set_ylabel("CO₂ Dual (EUR/t)")
    ax.set_title("Hourly CO₂ Dual (EUR/t)")
    # after you compute `data`
    first = data.index.min().normalize().replace(day=1)
    last = data.index.max()
    last_month_start = last.normalize().replace(day=1)
    next_month_start = last_month_start + pd.offsets.MonthBegin(1)

    ax.set_xlim(first, next_month_start)
    ax.margins(x=0)  # still no extra padding beyond these limits
    ax.margins(x=0)

    # Monthly ticks
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))
    ax.xaxis.set_minor_locator(NullLocator())

    ax.set_ylim(0, 120)

    ax.grid(axis='y', alpha=0.4)
    ax.set_axisbelow(True)

    
    # Legend below
    ax.legend(
        loc="upper center",
        bbox_to_anchor=(0.5, -0.12),
        ncol=len(scenarios) * 2,
        frameon=False,
    )

    fig.tight_layout()
    return fig


In [ ]:
# all_colors 
# aqua, cyan, turquoise, teal, green, lime, forestgreen, chartreuse,
# blue, royalblue, deepskyblue, dodgerblue, skyblue, cornflowerblue,
# red, firebrick, crimson, orange, darkorange, gold, purple, indigo, magenta,
# brown, sienna, chocolate, gray, dimgray, slategray, black, white, lavenderblush

scenario_colors = {
    "Baseline":  "darkred",
    "NoStorageCO2": "gold",
    "NoStorageALL": "gray",
    "H2":        "dodgerblue",
    "BioFixed":  "green",
    "CO2Trans": "purple"
}

start="2019-01-01"
end="2020-01-01"

fig = plot_scenarios(df, ["NoStorageCO2", "NoStorageALL"], scenario_colors, start=start, end=end)

fig.savefig("DualCO2Hourly.pdf", format="pdf", bbox_inches="tight")
plt.show(fig)


PLOTTING ONLY MOVING AVERAGES (MAs)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import NullLocator

def plot_ma_scenarios(df, scenarios, scenario_colors=None,
                      start=None, end=None, ma_window=168, min_per=1):
    """
    Plot only moving-average (MA) trends for selected scenarios.
    """

    if scenario_colors is None:
        scenario_colors = {}

    data = df[scenarios]
    if start or end:
        data = data.loc[start:end]

    fig, ax = plt.subplots(figsize=(14, 5))

    for scen in scenarios:
        ma = data[scen].rolling(window=ma_window, min_periods=min_per).mean()
        color = scenario_colors.get(scen, None)

        ax.plot(
            data.index,
            ma,
            linestyle="-",
            linewidth=2,
            label=scen,
            color=color,
        )

    ax.set_xlabel("")
    ax.set_ylabel("CO₂ Dual (EUR/t)")
    ax.set_title(f"{ma_window}-hour Moving Average CO₂ Dual")

    # Month-aligned x-limits
    first_month_start = data.index.min().normalize().replace(day=1)
    last_month_start = data.index.max().normalize().replace(day=1)
    next_month_start = last_month_start + pd.offsets.MonthBegin(1)

    ax.set_xlim(first_month_start, next_month_start)
    ax.margins(x=0)

    # Monthly ticks only
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))
    ax.xaxis.set_minor_locator(NullLocator())

    ax.grid(axis='y', alpha=0.4)
    ax.set_axisbelow(True)

    # Legend below
    legend = ax.legend(
        loc="upper center",
        bbox_to_anchor=(0.5, -0.12),
        ncol=len(scenarios),
        frameon=False
    )
    for line in legend.get_lines():
        line.set_linewidth(3)

    fig.tight_layout()
    return fig

scenario_colors = {
    "Baseline":     "darkred",
    "NoStorageCO2": "gold",
    "NoStorageALL": "gray",
    "H2Sales":      "dodgerblue",
    "BioFixed":     "green",
    "CO2Abundance": "purple"
}

start = "2019-01-01"
end = "2020-01-01"

fig = plot_ma_scenarios(
    df,
    ["Baseline", "NoStorageCO2", "NoStorageALL", "H2Sales", "CO2Abundance"],
    scenario_colors=scenario_colors,
    start=start,
    end=end,
    ma_window=168,
)

fig.savefig("DualCO2MA.pdf", format="pdf", bbox_inches="tight")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Increase height slightly (from 4 to 4.5) to give the legend room
fig, ax = plt.subplots(figsize=(10, 3))

scenarios = ["Baseline", "NoStorageCO2", "NoStorageALL", "H2Sales", "CO2Abundance"]

for s in scenarios:
    # Assuming df is defined in your environment
    x = np.sort(df[s].dropna().values)
    y = np.arange(1, len(x) + 1) / len(x)

    line, = ax.plot(x, y, label=s, linewidth=2)
    color = line.get_color()
    
    ax.vlines(
        x=x.max(),
        ymin=0,
        ymax=0.98,
        color=color,
        linestyle="--",
        linewidth=1.5,
        alpha=0.8
    )

ax.set_ylim(0, 1.02) # Slightly higher to avoid clipping the top of the curve
ax.set_xlim(0, 120)
ax.set_xticks(np.arange(0, 121, 10))
ax.set_xlabel("CO₂ dual (EUR/t)", fontsize=11)
ax.set_ylabel("Cumulative share of hours", fontsize=11)
ax.set_title("CDF of CO₂ dual values across scenarios", fontweight='bold', pad=15)

ax.grid(True, alpha=0.3)
ax.set_axisbelow(True)

# Fixed Legend Logic
legend = ax.legend(
    loc="upper center",
    bbox_to_anchor=(0.5, -0.15), # Adjusted anchor
    ncol=len(scenarios),
    fontsize=10,
    frameon=False,
    handletextpad=0.5,
    columnspacing=1.5
)

for line in legend.get_lines():
    line.set_linewidth(3)

# Use bbox_inches="tight" during save instead of complex tight_layout rects
plt.savefig("DualCO2_CDF.pdf", bbox_extra_artists=(legend,), bbox_inches="tight")
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# If df index is datetime, this will work nicely.
# If it's not datetime, it still plots but "weekly" won't be available.
s1, s2 = "Baseline", "NoStorageCO2"

fig, ax = plt.subplots(figsize=(12,4))

if np.issubdtype(df.index.dtype, np.datetime64):
    # weekly mean
    df[[s1, s2]].resample("W").mean().plot(ax=ax)
    ax.set_title("Weekly mean CO₂ dual (Baseline vs NoStorageCO2)")
else:
    # fallback: rolling mean over 168 hours (1 week)
    df[[s1, s2]].rolling(168, min_periods=1).mean().plot(ax=ax)
    ax.set_title("Rolling (168h) mean CO₂ dual (Baseline vs NoStorageCO2)")

ax.set_ylabel("CO₂ dual (EUR/t)")
ax.grid(True, alpha=0.3)
ax.legend()

plt.tight_layout()
plt.savefig("DualCO2_Time_Weekly.pdf", bbox_inches="tight")
plt.show()
